<a href="https://colab.research.google.com/github/1322vishav/Image-Processing/blob/main/Assignment11/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

np.random.seed(7)
x = np.random.randint(-5, 10, size=(3, 3)).astype(float)
k = np.random.randint(-2, 3, size=(3, 3)).astype(float)
print("Input matrix:\n", x)
print("Kernel:\n", k)

Input matrix:
 [[-1.  4.  1.]
 [-2. -2.  2.]
 [ 2.  7.  4.]]
Kernel:
 [[-2. -1.  0.]
 [ 0. -2.  2.]
 [-2.  2. -2.]]


In [2]:
def pad2d(a, p):
    return np.pad(a, p, mode='constant', constant_values=0)

def conv2d(a, kernel, pad=0, stride=1):
    a = pad2d(a, pad)
    kh, kw = kernel.shape
    oh = (a.shape[0] - kh) // stride + 1
    ow = (a.shape[1] - kw) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            patch = a[i * stride:i * stride + kh, j * stride:j * stride + kw]
            out[i, j] = np.sum(patch * kernel)
    return out

print("Padded (P=1):\n", pad2d(x, 1))
print("\nConv without padding:", conv2d(x, k, pad=0).shape, "\n", conv2d(x, k, pad=0))

bias = 0.5
conv_out = conv2d(x, k, pad=1) + bias
print("\nConv with P=1:", conv_out.shape, "\n", conv_out)

Padded (P=1):
 [[ 0.  0.  0.  0.  0.]
 [ 0. -1.  4.  1.  0.]
 [ 0. -2. -2.  2.  0.]
 [ 0.  2.  7.  4.  0.]
 [ 0.  0.  0.  0.  0.]]

Conv without padding: (1, 1) 
 [[8.]]

Conv with P=1: (3, 3) 
 [[ 10.5  -9.5   6.5]
 [ -8.5   8.5 -18.5]
 [ 12.5   0.5  -5.5]]


In [3]:
def relu(a):
    return np.maximum(0, a)

def sigmoid(a):
    return 1 / (1 + np.exp(-a))

relu_out = relu(conv_out)
sig_out = sigmoid(conv_out)
print("ReLU:\n", relu_out)
print("\nSigmoid:\n", np.round(sig_out, 4))

ReLU:
 [[10.5  0.   6.5]
 [ 0.   8.5  0. ]
 [12.5  0.5  0. ]]

Sigmoid:
 [[1.000e+00 1.000e-04 9.985e-01]
 [2.000e-04 9.998e-01 0.000e+00]
 [1.000e+00 6.225e-01 4.100e-03]]


In [4]:
def pool2d(a, size=2, stride=1, mode='max'):
    fn = np.max if mode == 'max' else np.min
    oh = (a.shape[0] - size) // stride + 1
    ow = (a.shape[1] - size) // stride + 1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = fn(a[i * stride:i * stride + size, j * stride:j * stride + size])
    return out

def global_avg_pool(a):
    return a.mean()

print("Max pooling:\n", pool2d(conv_out, mode='max'))
print("\nMin pooling:\n", pool2d(conv_out, mode='min'))
print("\nGlobal average pooling:", global_avg_pool(conv_out))

Max pooling:
 [[10.5  8.5]
 [12.5  8.5]]

Min pooling:
 [[ -9.5 -18.5]
 [ -8.5 -18.5]]

Global average pooling: -0.3888888888888889


In [5]:
flat = relu_out.flatten()
W = np.random.randn(flat.size, 4) * 0.5
b = np.zeros(4)
fc = flat @ W + b

print("Flattened:", flat)
print("Dense output:", np.round(fc, 4))

Flattened: [10.5  0.   6.5  0.   8.5  0.  12.5  0.5  0. ]
Dense output: [-10.3625 -13.4916  -1.1895 -12.5251]


In [6]:
def features(m):
    return relu(conv2d(m, k, pad=1) + bias).flatten()

batch = np.random.randint(-5, 10, size=(6, 3, 3)).astype(float)
dense_batch = np.array([features(m) @ W + b for m in batch])

def batch_norm(a, gamma, beta, eps=1e-5):
    mean = a.mean(axis=0)
    var = a.var(axis=0)
    return gamma * (a - mean) / np.sqrt(var + eps) + beta

bn = batch_norm(dense_batch, np.ones(4), np.zeros(4))
print("Before BN:\n", np.round(dense_batch, 3))
print("\nAfter BN:\n", np.round(bn, 3))
print("\nMean after BN:", np.round(bn.mean(axis=0), 4))
print("Variance after BN:", np.round(bn.var(axis=0), 4))

Before BN:
 [[ 13.07   12.567 -21.771  -1.598]
 [  6.673  -5.61    4.414  -6.871]
 [  7.329   6.351   1.004 -14.209]
 [  9.735  -2.598   0.623 -14.047]
 [  1.532   0.779  -0.988  -0.535]
 [-12.042 -11.735  -6.336 -13.746]]

After BN:
 [[ 1.069  1.595 -2.076  1.182]
 [ 0.282 -0.705  0.956  0.279]
 [ 0.363  0.809  0.561 -0.977]
 [ 0.659 -0.323  0.517 -0.95 ]
 [-0.351  0.104  0.331  1.364]
 [-2.022 -1.48  -0.289 -0.898]]

Mean after BN: [ 0. -0. -0.  0.]
Variance after BN: [1. 1. 1. 1.]


In [7]:
def dropout(a, rate, training=True):
    if not training:
        return a
    mask = (np.random.rand(*a.shape) >= rate).astype(float)
    return a * mask / (1 - rate)

print("Input:", np.round(fc, 4))
print("Dropout (training):", np.round(dropout(fc, 0.5, True), 4))
print("Dropout (inference):", np.round(dropout(fc, 0.5, False), 4))

Input: [-10.3625 -13.4916  -1.1895 -12.5251]
Dropout (training): [-20.7249 -26.9831  -0.      -0.    ]
Dropout (inference): [-10.3625 -13.4916  -1.1895 -12.5251]


In [8]:
def early_stopping(val_losses, patience=3):
    best = np.inf
    best_epoch = 0
    wait = 0
    for epoch, v in enumerate(val_losses, 1):
        if v < best:
            best, best_epoch, wait = v, epoch, 0
        else:
            wait += 1
            if wait >= patience:
                return epoch, best_epoch, best
    return len(val_losses), best_epoch, best

val_losses = [1.00, 0.82, 0.66, 0.60, 0.58, 0.59, 0.61, 0.64, 0.70, 0.75]
stopped, best_epoch, best = early_stopping(val_losses, patience=3)
print(f"Stopped at epoch {stopped}, best epoch {best_epoch}, best val loss {best}")

Stopped at epoch 8, best epoch 5, best val loss 0.58
